# 🧠 NLP Preprocessing Pipeline
### Production-Quality Text Cleaning Engine
---
> **Assignment**: Build a modular NLP preprocessing pipeline capable of handling real-world noisy text.  
> Each task below lives in its own cell — run them top to bottom.

---
## 📝 Task 1 — Conceptual Understanding
Answer the following questions in your own words.

In [ ]:
# ============================================================
# TASK 1 — Conceptual Understanding
# ============================================================

print("""
Q1. Difference between 'Love' and 'love' in NLP
------------------------------------------------
Computers compare characters byte-by-byte, so 'L' != 'l'.
Without lowercasing, bag-of-words / TF-IDF models treat them as
two separate vocabulary entries with different feature indices —
even though they carry identical meaning. Lowercasing (the first
normalisation step) merges both into 'love', reducing vocabulary
size and improving generalisation.

Q2. What happens if stopwords are NOT removed?
----------------------------------------------
  * They dominate TF-IDF vectors, drowning out meaningful words.
  * Vocabulary grows unnecessarily -> more memory + training time.
  * Similarity measures skewed toward function words, not topics.
  * Clustering / classification results become noisier.

Q3. Two scenarios where REMOVING stopwords is HARMFUL
------------------------------------------------------
1. Sentiment / negation:  'not good' -> remove 'not' -> 'good'
   Sentiment completely inverted! Negation words are critical.

2. Question-Answering:  'Who is the president?' -> 'president'
   Interrogative structure (who/what/when) is stripped, making it
   impossible to determine the expected answer type.

Q4. Stemming vs Lemmatization
-----------------------------
STEMMING  — Crude suffix-stripping. Fast, may produce non-words.
             'studies' -> 'studi',  'running' -> 'run'
             Tools: Porter Stemmer, Snowball Stemmer

LEMMATIZATION — POS-aware vocabulary lookup. Slower, always valid.
             'studies' -> 'study',  'better' -> 'good'
             Tools: spaCy, NLTK WordNetLemmatizer

  Aspect        | Stemming          | Lemmatization
  --------------|-------------------|--------------------
  Speed         | Fast              | Slower
  Accuracy      | Lower             | Higher
  Output        | May not be a word | Always a valid word
  POS-aware     | No                | Yes
  Best for      | Search indexing   | NLU / QA / Chatbots
""")


---
## ⚙️ Task 2 — Build the Advanced Preprocessing Function

In [ ]:
# ============================================================
# TASK 2 — Advanced Preprocessing Function
# ============================================================

import re

# Short words (<=2 chars) that carry meaning — never removed
MEANINGFUL_SHORT = {
    "no", "not", "ok", "go", "do", "me", "my",
    "we", "us", "he", "she", "it", "am", "is", "be", "by"
}

def preprocess_text(text: str) -> dict:
    """
    Clean and tokenise a raw text string.

    Steps: validate -> remove URLs -> remove emails -> strip emojis
           -> lowercase -> remove numbers -> collapse repeated chars
           -> remove punctuation -> tokenise -> filter short tokens

    Returns dict: original, tokens, clean_sentence, error
    """
    original = text

    # Edge case: empty / non-string
    if not isinstance(text, str) or text.strip() == "":
        return {"original": original, "tokens": [],
                "clean_sentence": "", "error": "Empty or invalid input"}

    # Edge case: emoji-only
    emoji_stripped = re.sub(
        r'[\U00010000-\U0010ffff\U0001F600-\U0001F64F'
        r'\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF'
        r'\U0001F1E0-\U0001F1FF\u2600-\u26FF\u2700-\u27BF]+',
        '', text).strip()
    if emoji_stripped == "":
        return {"original": original, "tokens": [],
                "clean_sentence": "", "error": "Emoji-only input"}

    # Edge case: number-only
    if re.fullmatch(r'[\d\s\.\,\+\-\/\(\)]+', text.strip()):
        return {"original": original, "tokens": [],
                "clean_sentence": "", "error": "Number-only input"}

    # 1. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 2. Remove email-like patterns
    text = re.sub(r'\S+@\S+\.\S+', '', text)

    # 3. Strip emojis / non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 4. Lowercase
    text = text.lower()

    # 5. Remove standalone numbers + noise symbols ($, %, #, @, *)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'[\$\€\£\%\#\@\*]+', '', text)

    # 6. Collapse 3+ repeated chars to 1  ('soooo'->'so', 'baaad'->'bad')
    #    Normal double letters ('free','good','happy') are preserved.
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 7. Remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # 8. Tokenise + remove extra whitespace
    raw_tokens = text.split()

    # 9. Drop tokens <=2 chars (keep MEANINGFUL_SHORT exceptions)
    tokens = [t for t in raw_tokens if len(t) > 2 or t in MEANINGFUL_SHORT]

    return {
        "original": original,
        "tokens": tokens,
        "clean_sentence": " ".join(tokens),
        "error": None
    }


# Quick feature-by-feature sanity check
test_cases = [
    ("I have 2 dogs",                "Remove numbers"),
    ("This  is   good",              "Remove extra spaces"),
    ("soooo goooood!!!",             "Repeated characters"),
    ("WOW!!! This is GREAT!!!",      "Lowercase + punctuation"),
    ("I am not OK",                  "Keep short meaningful words"),
    ("Visit http://example.com now", "Remove URL"),
    ("Email me at hi@test.com",      "Remove email"),
]

print(f"{'Input':<38} {'Cleaned':<30} {'Feature Tested'}")
print("-" * 85)
for text, feature in test_cases:
    r = preprocess_text(text)
    print(f"{text:<38} {r['clean_sentence']:<30} {feature}")


---
## 🧪 Task 3 — Stress Testing (10 Diverse Sentences)

In [ ]:
# ============================================================
# TASK 3 — Stress Testing
# ============================================================

SAMPLE_INPUTS = [
    "Get 100% FREE access now!!!",          # numbers + punctuation
    "I absolutely looooved this product 😍😍", # emojis + repeated chars
    "Worst service ever... 0/10",            # noise rating
    "Call me at 9876543210",                  # phone number
    "This is THE best course!!!",             # mixed case + punctuation
    "Visit https://openai.com now!",          # URL
    "Nooooo this is baaad!!!",               # repeated chars + slang
    "OK OK OK I got it",                     # short repeated word
    "Win $$$ now!!! Limited offer!!!",        # currency symbols
    "I am not happy with this"               # negation preservation test
]

stress_results = [preprocess_text(t) for t in SAMPLE_INPUTS]

for i, r in enumerate(stress_results, 1):
    print(f"{'─'*60}")
    print(f"  [{i:02d}] Original : {r['original']}")
    print(f"       Tokens   : {r['tokens']}")
    print(f"       Cleaned  : {r['clean_sentence']}")
print(f"{'─'*60}")


---
## 📊 Task 4 — Token Analytics

In [ ]:
# ============================================================
# TASK 4 — Token Analytics
# ============================================================

print(f"{'#':<4} {'Original':<42} {'Total':>6} {'Unique':>7} {'AvgLen':>7}")
print("─" * 70)

analytics = []
for i, r in enumerate(stress_results, 1):
    toks   = r["tokens"]
    total  = len(toks)
    unique = len(set(toks))
    avg    = round(sum(len(t) for t in toks) / total, 2) if total else 0
    analytics.append({"original": r["original"], "total": total,
                       "unique": unique, "avg": avg})
    short = r['original'][:40] + ('...' if len(r['original']) > 40 else '')
    print(f"  {i:<2} {short:<42} {total:>6} {unique:>7} {avg:>7}")

# Analysis
noisiest = min(analytics, key=lambda x: x['total'])
richest  = max(analytics, key=lambda x: x['total'])

print()
print("Analysis:")
print(f"  Most noisy (fewest tokens left)  -> '{noisiest['original']}'")
print(f"  ({noisiest['total']} token(s) retained after cleaning)")
print()
print(f"  Most meaningful tokens retained  -> '{richest['original']}'")
print(f"  ({richest['total']} tokens, avg length {richest['avg']} chars)")


---
## 🔢 Task 5 — Frequency Analysis

In [ ]:
# ============================================================
# TASK 5 — Frequency Analysis
# ============================================================

from collections import Counter

all_tokens = [tok for r in stress_results for tok in r["tokens"]]
freq = Counter(all_tokens)

print(f"Total tokens in corpus : {len(all_tokens)}")
print(f"Unique vocabulary size : {len(freq)}")

print("\nTop 10 Most Frequent Words")
print(f"  {'Word':<20} {'Count':>5}  Bar")
print("  " + "-" * 45)
for word, count in freq.most_common(10):
    bar = "#" * (count * 4)
    print(f"  {word:<20} {count:>5}  {bar}")

print("\nTop 5 Least Frequent Words")
print(f"  {'Word':<20} {'Count':>5}")
print("  " + "-" * 28)
for word, count in freq.most_common()[:-6:-1]:
    print(f"  {word:<20} {count:>5}")


---
## 🔧 Task 6 — Full Pipeline

In [ ]:
# ============================================================
# TASK 6 — Full Pipeline
# ============================================================

def full_pipeline(text_list: list) -> dict:
    """
    Run the complete NLP preprocessing pipeline on a list of strings.

    Returns
    -------
    dict:
        tokens           - list of token lists (one per sentence)
        clean_sentences  - list of cleaned sentence strings
        analytics        - per-sentence token statistics
        frequency        - corpus-level top-10 / least-5 word counts
    """
    if not isinstance(text_list, list) or len(text_list) == 0:
        return {"error": "Input must be a non-empty list of strings"}

    results = [preprocess_text(t) for t in text_list]

    # Per-sentence analytics
    analytics = []
    for r in results:
        toks = r["tokens"]
        n = len(toks)
        analytics.append({
            "original"        : r["original"],
            "total_tokens"    : n,
            "unique_tokens"   : len(set(toks)),
            "avg_token_length": round(sum(len(t) for t in toks)/n, 2) if n else 0
        })

    # Corpus frequency
    all_toks = [t for r in results for t in r["tokens"]]
    counter  = Counter(all_toks)

    return {
        "tokens"         : [r["tokens"] for r in results],
        "clean_sentences": [r["clean_sentence"] for r in results],
        "analytics"      : analytics,
        "frequency"      : {
            "top_10" : counter.most_common(10),
            "least_5": counter.most_common()[:-6:-1],
            "total"  : len(all_toks)
        }
    }


# Run the pipeline
output = full_pipeline(SAMPLE_INPUTS)

print("=" * 60)
print("OUTPUT STRUCTURE: {tokens, clean_sentences, analytics, frequency}")
print("=" * 60)

print("\ntokens (one list per sentence):")
for i, toks in enumerate(output["tokens"]):
    print(f"  [{i}] {toks}")

print("\nclean_sentences:")
for i, s in enumerate(output["clean_sentences"]):
    print(f"  [{i}] {s}")

print("\nanalytics:")
for a in output["analytics"]:
    print(f"  {a['original'][:45]:<46}"
          f"total={a['total_tokens']}  "
          f"unique={a['unique_tokens']}  "
          f"avg_len={a['avg_token_length']}")

print("\nfrequency:")
print("  top_10  :", output["frequency"]["top_10"])
print("  least_5 :", output["frequency"]["least_5"])
print("  total   :", output["frequency"]["total"])


---
## 🛡️ Task 7 — Error Handling

In [ ]:
# ============================================================
# TASK 7 — Error Handling
# ============================================================

edge_cases = [
    ("",             "Empty string"),
    ("😍🔥💯🎉",  "Only emojis"),
    ("1234 5678 90", "Only numbers"),
    (None,           "Non-string (None)"),
    ("   ",          "Only whitespace"),
]

print(f"{'Case':<22} {'Tokens':<10}  Error Message")
print("-" * 65)
for text, label in edge_cases:
    r = preprocess_text(text)
    print(f"  {label:<20} {str(r['tokens']):<10}  {r['error']}")

print()
print("All edge cases handled gracefully — no exceptions raised.")


---
## ✅ Summary

| Task | Description | Status |
|------|-------------|--------|
| 1 | Conceptual Understanding | ✅ |
| 2 | `preprocess_text()` function | ✅ |
| 3 | Stress testing (10 sentences) | ✅ |
| 4 | Token analytics per sentence | ✅ |
| 5 | Frequency analysis (top-10 / least-5) | ✅ |
| 6 | `full_pipeline()` with structured output | ✅ |
| 7 | Error handling (empty, emoji-only, number-only) | ✅ |

---
*Built with pure Python — `re`, `collections.Counter`. No external NLP libraries required.*